In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


In [ ]:
PROJECT_DIR = Path(str(project_path(get_value('science.paths.directory_008', 'directory_008'))))
STAGE2_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010') / get_value('science.paths.directory_011', 'directory_011')
if not STAGE2_DIR.exists():
    PROJECT_DIR = Path(str(project_path('')))
    STAGE2_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010') / get_value('science.paths.directory_011', 'directory_011')
SUMMARY_PATH = STAGE2_DIR / get_value('science.paths.directory_012', 'directory_012') / get_value('science.tables.metrics_009', 'metrics_009.csv')
OUT_DIR = STAGE2_DIR / get_value('science.paths.directory_108', 'directory_108')
OUT_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:
plt.rcParams.update({'font.family': 'Arial', 'font.size': 13.0, 'axes.titlesize': 15.2, 'axes.labelsize': 13.6, 'xtick.labelsize': 12.0, 'ytick.labelsize': 12.0, 'legend.fontsize': 11.8, 'axes.linewidth': 1.15, 'xtick.major.width': 1.15, 'ytick.major.width': 1.15, 'xtick.major.size': 4.6, 'ytick.major.size': 4.6, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'axes.grid': False, 'text.color': '#111111', 'axes.labelcolor': '#111111', 'xtick.color': '#111111', 'ytick.color': '#111111'})
MODALITY_COLORS = {'M': '#177D72', 'R': '#C65F45', 'B': '#7252A3', 'U': '#4F7F3C'}
ENDPOINT_COLORS = {'Response': '#C94C4C', 'PFS': '#2F6FA3', 'OS': '#6A4C93'}
COLOR_TEXT = '#111111'
COLOR_ERROR = '#424242'
COLOR_CONNECTOR = '#2F2F2F'
COLOR_FULL = '#2E2E2E'
ABSENT_FACE = '#FFFFFF'
ABSENT_EDGE = '#8C8C8C'
GRID_COLOR = '#C8C8C8'
SEPARATOR_COLOR = '#9A9A9A'
BG_MRU = '#D9EFE9'
BG_FULL = '#E7E7E7'
MRU_EDGE = '#084E46'
FIGSIZE = (14.4, 8.6)
COMMON_XLIM = (0.5, 0.96)
COMMON_XTICKS = [0.5, 0.6, 0.7, 0.8, 0.9]
GROUP_GAP = 0.65


In [ ]:
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f'\nCannot find:\n{SUMMARY_PATH}\n\nPlease run Stage2B in the latest modeling notebook first.')
df = pd.read_csv(SUMMARY_PATH)
required_cols = ['model', 'modalities', 'n_modalities', 'n_samples', 'n_features', 'Response_AUC_mean', 'Response_AUC_sd', 'PFS_Cindex_mean', 'PFS_Cindex_sd', 'OS_Cindex_mean', 'OS_Cindex_sd']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError('Missing columns:\n' + '\n'.join(missing))
df['n_modalities'] = pd.to_numeric(df['n_modalities'], errors='coerce').astype(int)
print('Models found:', len(df))
df.head()


In [ ]:
ABBR = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}

def short_modalities(x):
    mods = str(x).split('+')
    return '+'.join((ABBR.get(m, m) for m in mods))
df['short'] = df['modalities'].apply(short_modalities)
MODEL_ORDER = ['M', 'R', 'B', 'U', 'M+R', 'M+B', 'M+U', 'R+B', 'R+U', 'B+U', 'M+R+B', 'M+R+U', 'M+B+U', 'R+B+U', 'M+R+B+U']
order_map = {name: i for i, name in enumerate(MODEL_ORDER)}
df['plot_order'] = df['short'].map(order_map)
if df['plot_order'].isna().any():
    print(df.loc[df['plot_order'].isna(), ['model', 'modalities', 'short']])
    raise ValueError('Unexpected modality combination.')
df = df.sort_values('plot_order').reset_index(drop=True)
assert len(df) == 15, f'Expected 15 models, found {len(df)}.'
df[['short', 'n_modalities', 'Response_AUC_mean', 'PFS_Cindex_mean', 'OS_Cindex_mean']]


In [ ]:
row_y = {}
current_y = 0.0
for n_mod in [1, 2, 3, 4]:
    indexes = df.index[df['n_modalities'] == n_mod].tolist()
    for idx in indexes:
        row_y[idx] = current_y
        current_y += 1.0
    current_y += GROUP_GAP
df['raw_y'] = [row_y[i] for i in df.index]
max_y = df['raw_y'].max()
df['y'] = max_y - df['raw_y']
MRU = 'M+R+U'
FULL = 'M+R+B+U'
mru_tmp = df[df['short'] == MRU]
full_tmp = df[df['short'] == FULL]
if len(mru_tmp) != 1:
    raise ValueError('MRU was not found uniquely.')
if len(full_tmp) != 1:
    raise ValueError('Full multimodal model was not found uniquely.')
MRU_Y = float(mru_tmp['y'].iloc[0])
FULL_Y = float(full_tmp['y'].iloc[0])
print('MRU y:', MRU_Y)
print('Full y:', FULL_Y)


In [ ]:
fig = plt.figure(figsize=(21.8, 9.4), facecolor='white')
gs = GridSpec(1, 4, width_ratios=[2.35, 1.55, 1.55, 1.55], wspace=0.12)
ax_matrix = fig.add_subplot(gs[0, 0])
ax_resp = fig.add_subplot(gs[0, 1])
ax_pfs = fig.add_subplot(gs[0, 2], sharey=ax_resp)
ax_os = fig.add_subplot(gs[0, 3], sharey=ax_resp)
all_axes = [ax_matrix, ax_resp, ax_pfs, ax_os]
for ax in all_axes:
    ax.axhspan(MRU_Y - 0.42, MRU_Y + 0.42, color=BG_MRU, zorder=0)
    ax.axhspan(FULL_Y - 0.42, FULL_Y + 0.42, color=BG_FULL, zorder=0)
MODS = ['M', 'R', 'B', 'U']
x_mod = np.arange(len(MODS))
for _, row in df.iterrows():
    y = row['y']
    present = row['short'].split('+')
    ax_matrix.scatter(x_mod, [y] * len(MODS), s=52, facecolor=ABSENT_FACE, edgecolor=ABSENT_EDGE, linewidth=1.25, zorder=2)
    present_x = [MODS.index(m) for m in MODS if m in present]
    if len(present_x) > 1:
        ax_matrix.plot([min(present_x), max(present_x)], [y, y], color=COLOR_CONNECTOR, linewidth=2.3, zorder=2)
    for m in present:
        x = MODS.index(m)
        if row['short'] == MRU:
            size = 74
            edge = MRU_EDGE
            edge_width = 1.45
        elif row['short'] == FULL:
            size = 64
            edge = '#333333'
            edge_width = 1.2
        else:
            size = 56
            edge = 'white'
            edge_width = 0.85
        ax_matrix.scatter(x, y, s=size, facecolor=MODALITY_COLORS[m], edgecolor=edge, linewidth=edge_width, zorder=3)
    if row['short'] == MRU:
        label = 'M+R+U   MRU'
        weight = 'bold'
        text_color = MRU_EDGE
    elif row['short'] == FULL:
        label = 'M+R+B+U   Full'
        weight = 'bold'
        text_color = COLOR_FULL
    else:
        label = row['short']
        weight = 'normal'
        text_color = COLOR_TEXT
    ax_matrix.text(-0.84, y, label, ha='right', va='center', fontsize=13.8, fontweight=weight, color=text_color)
ax_matrix.set_xlim(-2.95, 3.52)
ax_matrix.set_ylim(df['y'].min() - 0.75, df['y'].max() + 0.75)
ax_matrix.set_xticks(x_mod)
ax_matrix.set_xticklabels(MODS, fontweight='bold', fontsize=15.0)
for tick, m in zip(ax_matrix.get_xticklabels(), MODS):
    tick.set_color(MODALITY_COLORS[m])
ax_matrix.xaxis.tick_top()
ax_matrix.tick_params(axis='x', length=0, pad=10.5)
ax_matrix.set_yticks([])
for spine in ax_matrix.spines.values():
    spine.set_visible(False)
metric_specs = [{'ax': ax_resp, 'mean': 'Response_AUC_mean', 'sd': 'Response_AUC_sd', 'title': 'Response', 'xlabel': 'AUC', 'color': ENDPOINT_COLORS['Response']}, {'ax': ax_pfs, 'mean': 'PFS_Cindex_mean', 'sd': 'PFS_Cindex_sd', 'title': 'PFS', 'xlabel': 'C-index', 'color': ENDPOINT_COLORS['PFS']}, {'ax': ax_os, 'mean': 'OS_Cindex_mean', 'sd': 'OS_Cindex_sd', 'title': 'OS', 'xlabel': 'C-index', 'color': ENDPOINT_COLORS['OS']}]
for spec in metric_specs:
    ax = spec['ax']
    endpoint_color = spec['color']
    for _, row in df.iterrows():
        y = row['y']
        mean = float(row[spec['mean']])
        sd = float(row[spec['sd']])
        is_mru = row['short'] == MRU
        is_full = row['short'] == FULL
        if is_mru:
            marker = 'D'
            ms = 9.4
            alpha = 1.0
            edgecolor = MRU_EDGE
            edgewidth = 1.15
            z = 6
        elif is_full:
            marker = 's'
            ms = 8.4
            alpha = 0.9
            edgecolor = '#333333'
            edgewidth = 1.15
            z = 5
        else:
            marker = 'o'
            ms = 6.8
            alpha = 0.88
            edgecolor = 'white'
            edgewidth = 0.7
            z = 3
        ax.errorbar(mean, y, xerr=sd, fmt='none', ecolor=endpoint_color if is_mru or is_full else COLOR_ERROR, elinewidth=1.85, capsize=3.8, capthick=1.5, alpha=0.98 if is_mru or is_full else 0.82, zorder=z - 1)
        ax.plot(mean, y, marker=marker, markersize=ms, linestyle='None', markerfacecolor=endpoint_color, markeredgecolor=edgecolor, markeredgewidth=edgewidth, alpha=alpha, zorder=z)
        if is_mru or is_full:
            annot_x = mean + sd + 0.01
            annot_ha = 'left'
            if annot_x > COMMON_XLIM[1] - 0.035:
                annot_x = max(COMMON_XLIM[0] + 0.02, mean - sd - 0.012)
                annot_ha = 'right'
            ax.text(annot_x, y, f'{mean:.3f}', ha=annot_ha, va='center', fontsize=12.8, color=MRU_EDGE if is_mru else COLOR_FULL, fontweight='bold' if is_mru else 'normal')
    ax.set_xlim(COMMON_XLIM)
    ax.set_xticks(COMMON_XTICKS)
    ax.set_xlabel(spec['xlabel'])
    ax.set_title(spec['title'], pad=13.5, fontweight='bold', color=endpoint_color)
    ax.tick_params(axis='y', left=False, labelleft=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
for n_mod in [1, 2, 3]:
    current_group = df.loc[df['n_modalities'] == n_mod, 'y'].to_numpy()
    next_group = df.loc[df['n_modalities'] == n_mod + 1, 'y'].to_numpy()
    y_sep = (current_group.min() + next_group.max()) / 2
    for ax in all_axes:
        ax.axhline(y_sep, color=SEPARATOR_COLOR, linewidth=1.1, zorder=1)
for n_mod in [1, 2, 3, 4]:
    ys = df.loc[df['n_modalities'] == n_mod, 'y'].to_numpy()
    y_center = ys.mean()
    text = '1 modality' if n_mod == 1 else f'{n_mod} modalities'
    ax_matrix.text(-2.74, y_center, text, rotation=90, ha='center', va='center', fontsize=13.8, color='#3F3F3F', fontweight='bold')
fig.text(0.555, 0.948, '◆', color=ENDPOINT_COLORS['PFS'], fontsize=14.6, va='center')
fig.text(0.57, 0.948, 'MRU', fontsize=12.8, va='center')
fig.text(0.617, 0.948, '■', color=COLOR_FULL, fontsize=13.8, va='center')
fig.text(0.632, 0.948, 'Full', fontsize=12.8, va='center')
fig.subplots_adjust(left=0.065, right=0.985, top=0.91, bottom=0.085)
STEM = OUT_DIR / get_value('science.paths.directory_109', 'directory_109')
fig.savefig(str(STEM) + '.png', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()






In [ ]:
source_cols = ['model', 'modalities', 'short', 'n_modalities', 'n_samples', 'n_features', 'Response_AUC_mean', 'Response_AUC_sd', 'PFS_Cindex_mean', 'PFS_Cindex_sd', 'OS_Cindex_mean', 'OS_Cindex_sd']
source_path = OUT_DIR / get_value('science.tables.result_189', 'result_189.csv')
df[source_cols].to_csv(source_path, index=False, encoding='utf-8-sig')




In [ ]:
BASE_FOLD_PATH = STAGE2_DIR / get_value('science.tables.metrics_005', 'metrics_005.csv')
COMBO_FOLD_PATH = STAGE2_DIR / get_value('science.paths.directory_012', 'directory_012') / get_value('science.tables.metrics_008', 'metrics_008.csv')
if not BASE_FOLD_PATH.exists():
    raise FileNotFoundError(f'Cannot find:\n{BASE_FOLD_PATH}')
if not COMBO_FOLD_PATH.exists():
    raise FileNotFoundError(f'Cannot find:\n{COMBO_FOLD_PATH}')
fold_base = pd.read_csv(BASE_FOLD_PATH)
fold_combo = pd.read_csv(COMBO_FOLD_PATH)


print('\nBase fold shape :', fold_base.shape)
print('Combo fold shape:', fold_combo.shape)


In [ ]:
model_metadata = df[['model', 'modalities', 'n_modalities', 'short']].drop_duplicates('model').copy()
fold_base_4b = fold_base.copy()
for col in ['modalities', 'n_modalities', 'short']:
    if col in fold_base_4b.columns:
        fold_base_4b = fold_base_4b.drop(columns=col)
fold_base_4b = fold_base_4b.merge(model_metadata, on='model', how='left')
fold_combo_4b = fold_combo.copy()
for col in ['modalities', 'n_modalities', 'short']:
    if col in fold_combo_4b.columns:
        fold_combo_4b = fold_combo_4b.drop(columns=col)
fold_combo_4b = fold_combo_4b.merge(model_metadata, on='model', how='left')
fold_all_4b = pd.concat([fold_base_4b, fold_combo_4b], axis=0, ignore_index=True)
required_fold_cols = ['model', 'fold_id', 'short', 'Response_AUC', 'PFS_Cindex', 'OS_Cindex']
missing_fold_cols = [c for c in required_fold_cols if c not in fold_all_4b.columns]
if missing_fold_cols:
    raise ValueError('Missing fold-level columns:\n' + '\n'.join(missing_fold_cols))
print(fold_all_4b.groupby('short')['fold_id'].nunique().sort_index())


In [ ]:
REMOVE_MAP = {'M': 'R+B+U', 'R': 'M+B+U', 'B': 'M+R+U', 'U': 'M+R+B'}
REMOVE_ORDER = ['M', 'R', 'B', 'U']
FOLD_METRICS = {'Response': 'Response_AUC', 'PFS': 'PFS_Cindex', 'OS': 'OS_Cindex'}
full_fold = fold_all_4b[fold_all_4b['short'] == FULL][['fold_id', 'Response_AUC', 'PFS_Cindex', 'OS_Cindex']].copy()
if full_fold.empty:
    raise ValueError('Full model fold-level results were not found.')
if full_fold['fold_id'].duplicated().any():
    raise ValueError('Duplicate fold_id values detected for Full model.')
paired_records = []
for removed_modality in REMOVE_ORDER:
    loo_short = REMOVE_MAP[removed_modality]
    loo_fold = fold_all_4b[fold_all_4b['short'] == loo_short][['fold_id', 'Response_AUC', 'PFS_Cindex', 'OS_Cindex']].copy()
    if loo_fold.empty:
        raise ValueError(f'Fold-level results not found for {loo_short}.')
    if loo_fold['fold_id'].duplicated().any():
        raise ValueError(f'Duplicate fold_id values detected for {loo_short}.')
    paired = loo_fold.merge(full_fold, on='fold_id', how='inner', suffixes=('_LOO', '_Full'))
    print(f'Remove {removed_modality}: {loo_short} vs Full -> {len(paired)} paired folds')
    for _, row in paired.iterrows():
        for endpoint, metric in FOLD_METRICS.items():
            paired_records.append({'removed_modality': removed_modality, 'leave_one_out_model': loo_short, 'fold_id': int(row['fold_id']), 'endpoint': endpoint, 'LOO_performance': float(row[f'{metric}_LOO']), 'Full_performance': float(row[f'{metric}_Full']), 'delta': float(row[f'{metric}_LOO'] - row[f'{metric}_Full'])})
delta_long = pd.DataFrame(paired_records)
delta_long.head()


In [ ]:
delta_summary = delta_long.groupby(['removed_modality', 'leave_one_out_model', 'endpoint'], as_index=False).agg(mean_delta=('delta', 'mean'), sd_delta=('delta', 'std'), median_delta=('delta', 'median'), min_delta=('delta', 'min'), max_delta=('delta', 'max'), n_paired_folds=('delta', 'size'))
delta_summary['removed_order'] = delta_summary['removed_modality'].map({m: i for i, m in enumerate(REMOVE_ORDER)})
delta_summary['endpoint_order'] = delta_summary['endpoint'].map({'Response': 0, 'PFS': 1, 'OS': 2})
delta_summary = delta_summary.sort_values(['removed_order', 'endpoint_order']).reset_index(drop=True)
delta_summary


In [ ]:
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.patches import Rectangle
row_order = ['M', 'R', 'B', 'U']
endpoint_order = ['Response', 'PFS', 'OS']
heat = delta_summary.pivot(index='removed_modality', columns='endpoint', values='mean_delta').reindex(index=row_order, columns=endpoint_order)
heat_sd = delta_summary.pivot(index='removed_modality', columns='endpoint', values='sd_delta').reindex(index=row_order, columns=endpoint_order)
heat_n = delta_summary.pivot(index='removed_modality', columns='endpoint', values='n_paired_folds').reindex(index=row_order, columns=endpoint_order)
print('Mean paired Δperformance:')
display(heat)
print('\nSD across paired CV-fold differences:')
display(heat_sd)
max_abs = np.nanmax(np.abs(heat.to_numpy(dtype=float)))
COLOR_STEP = 0.01
COLOR_LIMIT = np.ceil(max_abs / COLOR_STEP) * COLOR_STEP
COLOR_LIMIT = max(COLOR_LIMIT, 0.03)
print('Heatmap color range:', f'{-COLOR_LIMIT:.3f} to {COLOR_LIMIT:.3f}')
ABLATION_CMAP = LinearSegmentedColormap.from_list('ablation_diverging', ['#2B6CB0', '#A9C8E4', '#F7F7F7', '#F3B59E', '#C84E40'], N=256)
norm = TwoSlopeNorm(vmin=-COLOR_LIMIT, vcenter=0.0, vmax=COLOR_LIMIT)
remaining_models = {'M': 'R+B+U', 'R': 'M+B+U', 'B': 'M+R+U', 'U': 'M+R+B'}
removed_full_names = {'M': 'Metagenomics', 'R': 'Radiomics', 'B': 'Blood', 'U': 'Urine'}
row_labels = []
for m in row_order:
    if m == 'B':
        row_labels.append('−B  →  M+R+U (MRU)')
    else:
        row_labels.append(f'−{m}  →  {remaining_models[m]}')
fig, ax = plt.subplots(figsize=(10.4, 6.1), facecolor='white')
matrix = heat.to_numpy(dtype=float)
im = ax.imshow(matrix, cmap=ABLATION_CMAP, norm=norm, aspect='auto', interpolation='nearest')
ax.set_xticks(np.arange(len(endpoint_order)))
ax.set_xticklabels(endpoint_order, fontweight='bold', fontsize=13.4)
for tick, endpoint in zip(ax.get_xticklabels(), endpoint_order):
    tick.set_color(ENDPOINT_COLORS[endpoint])
ax.xaxis.tick_top()
ax.tick_params(axis='x', top=True, bottom=False, labeltop=True, labelbottom=False, length=0, pad=10)
ax.set_yticks(np.arange(len(row_order)))
ax.set_yticklabels(row_labels, fontsize=12.8)
ax.tick_params(axis='y', length=0, pad=10)
for tick, m in zip(ax.get_yticklabels(), row_order):
    tick.set_color(MODALITY_COLORS[m])
    if m == 'B':
        tick.set_fontweight('bold')
for i, removed_modality in enumerate(row_order):
    for j, endpoint in enumerate(endpoint_order):
        value = float(heat.loc[removed_modality, endpoint])
        sd_value = float(heat_sd.loc[removed_modality, endpoint])
        normalized_strength = abs(value) / COLOR_LIMIT if COLOR_LIMIT > 0 else 0
        text_color = 'white' if normalized_strength > 0.6 else '#2F2F2F'
        weight = 'bold' if removed_modality == 'B' else 'normal'
        ax.text(j, i - 0.06, f'{value:+.3f}', ha='center', va='center', fontsize=12.6, fontweight=weight, color=text_color)
        ax.text(j, i + 0.18, f'SD {sd_value:.3f}', ha='center', va='center', fontsize=10.8, color=text_color, alpha=0.97)
ax.set_xticks(np.arange(-0.5, len(endpoint_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
ax.grid(which='minor', color='white', linestyle='-', linewidth=2.5)
ax.tick_params(which='minor', bottom=False, left=False)
mru_row_idx = row_order.index('B')
mru_rect = Rectangle((-0.5, mru_row_idx - 0.5), width=len(endpoint_order), height=1.0, fill=False, edgecolor=MRU_EDGE, linewidth=2.5, zorder=5)
ax.add_patch(mru_rect)
ax.text(len(endpoint_order) - 0.02, mru_row_idx, ' selected', ha='left', va='center', fontsize=11.4, fontweight='bold', color=MRU_EDGE, clip_on=False)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.055, pad=0.15, aspect=35)
cbar.set_label('Δ performance versus Full', fontsize=11.8)
cbar.ax.tick_params(labelsize=10.2, length=2.5)
cbar.outline.set_visible(False)
cbar.ax.text(0.0, -1.75, 'Removal decreases performance', transform=cbar.ax.transAxes, ha='left', va='top', fontsize=11.0, color='#4A4A4A')
cbar.ax.text(1.0, -1.75, 'Removal increases performance', transform=cbar.ax.transAxes, ha='right', va='top', fontsize=11.0, color='#4A4A4A')
fig.subplots_adjust(left=0.34, right=0.93, top=0.96, bottom=0.18)
STEM_4B = OUT_DIR / get_value('science.paths.directory_110', 'directory_110')
fig.savefig(str(STEM_4B) + '.png', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_4B) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_4B) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_4B) + '.png')
print(str(STEM_4B) + '.pdf')
print(str(STEM_4B) + '.svg')


In [ ]:
delta_long_path = OUT_DIR / get_value('science.tables.result_190', 'result_190.csv')
delta_long.to_csv(delta_long_path, index=False, encoding='utf-8-sig')
delta_summary_path = OUT_DIR / get_value('science.tables.metrics_067', 'metrics_067.csv')
delta_summary.to_csv(delta_summary_path, index=False, encoding='utf-8-sig')





In [ ]:
DEPTH_METRICS = {'Response': {'mean_col': 'Response_AUC_mean', 'ylabel': 'AUC', 'color': ENDPOINT_COLORS['Response']}, 'PFS': {'mean_col': 'PFS_Cindex_mean', 'ylabel': 'C-index', 'color': ENDPOINT_COLORS['PFS']}, 'OS': {'mean_col': 'OS_Cindex_mean', 'ylabel': 'C-index', 'color': ENDPOINT_COLORS['OS']}}
depth_records = []
for endpoint, cfg in DEPTH_METRICS.items():
    metric_col = cfg['mean_col']
    for n_mod in [1, 2, 3, 4]:
        sub = df[df['n_modalities'] == n_mod].copy()
        values = sub[metric_col].astype(float).to_numpy()
        best_idx = sub[metric_col].idxmax()
        best_row = df.loc[best_idx]
        depth_records.append({'endpoint': endpoint, 'n_modalities': n_mod, 'n_models': len(sub), 'mean_performance': np.mean(values), 'median_performance': np.median(values), 'min_performance': np.min(values), 'max_performance': np.max(values), 'sd_across_models': np.std(values, ddof=1) if len(values) > 1 else np.nan, 'best_model': best_row['short'], 'best_performance': float(best_row[metric_col])})
depth_summary = pd.DataFrame(depth_records)
depth_summary


In [ ]:
def depth_jitter(n, width=0.13):
    if n <= 1:
        return np.array([0.0])
    return np.linspace(-width, width, n)
fig, axes = plt.subplots(3, 1, figsize=(5.6, 11.8), sharex=True, facecolor='white')
endpoint_order_4c = ['Response', 'PFS', 'OS']
metric_cols_4c = {'Response': 'Response_AUC_mean', 'PFS': 'PFS_Cindex_mean', 'OS': 'OS_Cindex_mean'}
ylabs_4c = {'Response': 'AUC', 'PFS': 'C-index', 'OS': 'C-index'}
for ax, endpoint in zip(axes, endpoint_order_4c):
    color = ENDPOINT_COLORS[endpoint]
    metric_col = metric_cols_4c[endpoint]
    endpoint_depth = depth_summary[depth_summary['endpoint'] == endpoint].sort_values('n_modalities').copy()
    ax.grid(False)
    for _, row in endpoint_depth.iterrows():
        x = int(row['n_modalities'])
        ymin = float(row['min_performance'])
        ymax = float(row['max_performance'])
        ax.plot([x, x], [ymin, ymax], color=color, linewidth=7.0, alpha=0.24, solid_capstyle='round', zorder=1)
    for n_mod in [1, 2, 3, 4]:
        sub = df[df['n_modalities'] == n_mod].sort_values('short').copy()
        vals = sub[metric_col].astype(float).to_numpy()
        jit = depth_jitter(len(sub), width=0.13)
        for dx, (_, row) in zip(jit, sub.iterrows()):
            is_mru = row['short'] == MRU
            is_full = row['short'] == FULL
            if is_mru:
                marker = 'D'
                size = 64
                alpha = 1.0
                edge = MRU_EDGE
                edge_width = 1.15
                z = 6
            elif is_full:
                marker = 's'
                size = 58
                alpha = 0.95
                edge = '#333333'
                edge_width = 0.8
                z = 6
            else:
                marker = 'o'
                size = 34
                alpha = 0.55
                edge = 'white'
                edge_width = 0.55
                z = 3
            ax.scatter(n_mod + dx, float(row[metric_col]), s=size, marker=marker, facecolor=color, edgecolor=edge, linewidth=edge_width, alpha=alpha, zorder=z)
            if is_mru:
                ax.annotate('MRU', xy=(n_mod + dx, float(row[metric_col])), xytext=(0, 12), textcoords='offset points', ha='center', va='bottom', fontsize=11.4, fontweight='bold', color=MRU_EDGE, zorder=7)
            elif is_full:
                ax.annotate('Full', xy=(n_mod + dx, float(row[metric_col])), xytext=(0, 12), textcoords='offset points', ha='center', va='bottom', fontsize=10.8, color=COLOR_FULL, zorder=7)
    x_mean = endpoint_depth['n_modalities'].astype(float).to_numpy()
    y_mean = endpoint_depth['mean_performance'].astype(float).to_numpy()
    ax.plot(x_mean, y_mean, color=color, linewidth=2.2, marker='o', markersize=6.7, markerfacecolor='white', markeredgecolor=color, markeredgewidth=1.6, zorder=5)
    ax.set_title(endpoint, color=color, fontweight='bold', pad=11.5)
    ax.set_ylabel(ylabs_4c[endpoint])
    ax.set_xticks([1, 2, 3, 4])
    ax.set_xticklabels(['1', '2', '3', '4'], fontsize=12.2)
    if endpoint == endpoint_order_4c[-1]:
        ax.set_xlabel('No. of integrated modalities')
    else:
        ax.set_xlabel('')
    endpoint_values = df[metric_col].astype(float).to_numpy()
    ymin = max(0.45, endpoint_values.min() - 0.055)
    ymax = min(1.0, endpoint_values.max() + 0.075)
    ax.set_ylim(ymin, ymax)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out')
fig.subplots_adjust(left=0.2, right=0.98, top=0.97, bottom=0.08, hspace=0.34)
STEM_4C = OUT_DIR / get_value('science.paths.directory_111', 'directory_111')
fig.savefig(str(STEM_4C) + '.png', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_4C) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_4C) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_4C) + '.png')
print(str(STEM_4C) + '.pdf')
print(str(STEM_4C) + '.svg')


In [ ]:
depth_summary_path = OUT_DIR / get_value('science.tables.metrics_068', 'metrics_068.csv')
depth_summary.to_csv(depth_summary_path, index=False, encoding='utf-8-sig')
depth_model_source_path = OUT_DIR / get_value('science.tables.result_191', 'result_191.csv')
df[['model', 'modalities', 'short', 'n_modalities', 'Response_AUC_mean', 'PFS_Cindex_mean', 'OS_Cindex_mean']].to_csv(depth_model_source_path, index=False, encoding='utf-8-sig')





In [ ]:
MODEL_RESULTS_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010')
EXTERNAL_OUT_DIR = MODEL_RESULTS_DIR / get_value('science.paths.directory_050', 'directory_050')
EXTERNAL_LOCKED_PATH = EXTERNAL_OUT_DIR / get_value('science.tables.metrics_024', 'metrics_024.csv')
EXTERNAL_SUMMARY_PATH = EXTERNAL_OUT_DIR / get_value('science.tables.metrics_022', 'metrics_022.csv')
INTERNAL_EXTERNAL_PATH = EXTERNAL_OUT_DIR / get_value('science.tables.metrics_023', 'metrics_023.csv')
INTERNAL_SINGLE_PATH = STAGE2_DIR / get_value('science.tables.metrics_004', 'metrics_004.csv')


print('\nPreferred comparison table:')



In [ ]:
if INTERNAL_EXTERNAL_PATH.exists():
    external_compare = pd.read_csv(INTERNAL_EXTERNAL_PATH, encoding='utf-8-sig')
    
else:
    print(get_value('science.messages.status_027', 'input not found.'))
    print('Building the comparison table from locked source files...')
    if EXTERNAL_LOCKED_PATH.exists():
        external_locked = pd.read_csv(EXTERNAL_LOCKED_PATH, encoding='utf-8-sig')
    elif EXTERNAL_SUMMARY_PATH.exists():
        external_locked = pd.read_csv(EXTERNAL_SUMMARY_PATH, encoding='utf-8-sig')
    else:
        raise FileNotFoundError(f"\nCould not find either:\n{EXTERNAL_LOCKED_PATH}\nor\n{EXTERNAL_SUMMARY_PATH}\n\nPlease run the latest fixed external-validation notebook with RUN_MODALITY='all' first.")
    if not INTERNAL_SINGLE_PATH.exists():
        raise FileNotFoundError(f'Cannot find internal Stage2 summary:\n{INTERNAL_SINGLE_PATH}')
    internal_single = pd.read_csv(INTERNAL_SINGLE_PATH, encoding='utf-8-sig')
    internal_keep = ['model', 'Response_AUC_mean', 'Response_AUC_sd', 'Response_AUPRC_mean', 'Response_AUPRC_sd', 'PFS_Cindex_mean', 'PFS_Cindex_sd', 'OS_Cindex_mean', 'OS_Cindex_sd']
    internal_keep = [c for c in internal_keep if c in internal_single.columns]
    internal_single = internal_single[internal_keep].copy()
    external_compare = internal_single.merge(external_locked, on='model', how='inner')
required_4d_cols = ['model', 'Response_AUC_mean', 'Response_AUC_sd', 'PFS_Cindex_mean', 'PFS_Cindex_sd', 'OS_Cindex_mean', 'OS_Cindex_sd', 'n_external', 'Response_AUROC', 'Response_AUROC_CI_lower', 'Response_AUROC_CI_upper', 'PFS_Cindex', 'PFS_Cindex_CI_lower', 'PFS_Cindex_CI_upper', 'OS_Cindex', 'OS_Cindex_CI_lower', 'OS_Cindex_CI_upper']
missing_4d = [c for c in required_4d_cols if c not in external_compare.columns]
if missing_4d:
    raise ValueError('Figure 4D is missing required columns:\n' + '\n'.join(missing_4d))
print('\nAvailable external-validation models:')
display(external_compare[['model', 'n_external', 'Response_AUROC', 'PFS_Cindex', 'OS_Cindex']])


In [ ]:
EXTERNAL_MODEL_ORDER = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
EXTERNAL_SHORT = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}
EXTERNAL_DISPLAY = {'Metagenomics': 'Metagenomics (KO)', 'Radiomics': 'CT radiomics', 'Blood': 'Blood metabolomics', 'Urine': 'Urine metabolomics'}
external_compare = external_compare[external_compare['model'].isin(EXTERNAL_MODEL_ORDER)].copy()
external_compare['model_order'] = external_compare['model'].map({m: i for i, m in enumerate(EXTERNAL_MODEL_ORDER)})
external_compare = external_compare.sort_values('model_order').reset_index(drop=True)
external_compare['short'] = external_compare['model'].map(EXTERNAL_SHORT)
if 'display_name' in external_compare.columns:
    external_compare['display_name_4d'] = external_compare['display_name'].fillna(external_compare['model'].map(EXTERNAL_DISPLAY))
else:
    external_compare['display_name_4d'] = external_compare['model'].map(EXTERNAL_DISPLAY)
external_compare['row_label'] = external_compare['display_name_4d'] + '\n' + '(n=' + external_compare['n_external'].astype(int).astype(str) + ')'
display(external_compare[['model', 'short', 'row_label', 'n_external']])


In [ ]:
from matplotlib.patches import Rectangle
TRANSFER_SPECS = [{'endpoint': 'Response', 'title': 'Response', 'xlabel': 'AUROC', 'internal': 'Response_AUC_mean', 'internal_sd': 'Response_AUC_sd', 'external': 'Response_AUROC', 'external_lo': 'Response_AUROC_CI_lower', 'external_hi': 'Response_AUROC_CI_upper', 'color': ENDPOINT_COLORS['Response']}, {'endpoint': 'PFS', 'title': 'PFS', 'xlabel': 'C-index', 'internal': 'PFS_Cindex_mean', 'internal_sd': 'PFS_Cindex_sd', 'external': 'PFS_Cindex', 'external_lo': 'PFS_Cindex_CI_lower', 'external_hi': 'PFS_Cindex_CI_upper', 'color': ENDPOINT_COLORS['PFS']}, {'endpoint': 'OS', 'title': 'OS', 'xlabel': 'C-index', 'internal': 'OS_Cindex_mean', 'internal_sd': 'OS_Cindex_sd', 'external': 'OS_Cindex', 'external_lo': 'OS_Cindex_CI_lower', 'external_hi': 'OS_Cindex_CI_upper', 'color': ENDPOINT_COLORS['OS']}]
all_bounds = []
for spec in TRANSFER_SPECS:
    all_bounds.extend(external_compare[spec['external_lo']].astype(float).dropna().tolist())
    all_bounds.extend(external_compare[spec['external_hi']].astype(float).dropna().tolist())
    internal_low = external_compare[spec['internal']].astype(float) - external_compare[spec['internal_sd']].astype(float)
    internal_high = external_compare[spec['internal']].astype(float) + external_compare[spec['internal_sd']].astype(float)
    all_bounds.extend(internal_low.dropna().tolist())
    all_bounds.extend(internal_high.dropna().tolist())
raw_min = min(all_bounds)
raw_max = max(all_bounds)
XMIN_4D = 0.5
XMAX_4D = min(1.0, max(0.85, np.ceil((raw_max + 0.035) * 20) / 20))
print('Figure 4D common x-axis:', XMIN_4D, 'to', XMAX_4D)
fig, axes = plt.subplots(1, 3, figsize=(14.8, 5.8), sharey=True, facecolor='white')
y_positions_4d = np.arange(len(external_compare))[::-1]
BAR_HEIGHT = 0.34
INTERNAL_BAND_HEIGHT = 0.6
for ax, spec in zip(axes, TRANSFER_SPECS):
    endpoint_color = spec['color']
    ax.axvline(0.5, color='#666666', linestyle='--', linewidth=0.9, zorder=0)
    ax.grid(False)
    for y, (_, row) in zip(y_positions_4d, external_compare.iterrows()):
        internal_value = float(row[spec['internal']])
        internal_sd = float(row[spec['internal_sd']])
        external_value = float(row[spec['external']])
        external_lo = float(row[spec['external_lo']])
        external_hi = float(row[spec['external_hi']])
        modality_short = row['short']
        modality_color = MODALITY_COLORS[modality_short]
        ax.barh(y, XMAX_4D - XMIN_4D, left=XMIN_4D, height=BAR_HEIGHT, color='#F2F2F2', edgecolor='none', zorder=1)
        internal_lo = max(XMIN_4D, internal_value - internal_sd)
        internal_hi = min(XMAX_4D, internal_value + internal_sd)
        internal_band = Rectangle((internal_lo, y - INTERNAL_BAND_HEIGHT / 2), internal_hi - internal_lo, INTERNAL_BAND_HEIGHT, facecolor=modality_color, edgecolor='none', alpha=0.18, zorder=1.5)
        ax.add_patch(internal_band)
        external_bar_width = max(0.0, external_value - XMIN_4D)
        ax.barh(y, external_bar_width, left=XMIN_4D, height=BAR_HEIGHT, color=endpoint_color, alpha=0.96, edgecolor='none', zorder=2)
        ax.plot([max(XMIN_4D, external_lo), min(XMAX_4D, external_hi)], [y, y], color=endpoint_color, linewidth=1.85, zorder=4)
        ax.plot([max(XMIN_4D, external_lo), max(XMIN_4D, external_lo)], [y - 0.11, y + 0.11], color=endpoint_color, linewidth=1.35, zorder=4)
        ax.plot([min(XMAX_4D, external_hi), min(XMAX_4D, external_hi)], [y - 0.11, y + 0.11], color=endpoint_color, linewidth=1.35, zorder=4)
        ax.scatter(external_value, y, marker='s', s=42, facecolor=endpoint_color, edgecolor='white', linewidth=0.8, zorder=5)
        ax.plot([internal_value, internal_value], [y - 0.26, y + 0.26], color=modality_color, linewidth=2.35, solid_capstyle='round', zorder=6)
        label_x = min(XMAX_4D - 0.01, external_value + 0.012)
        if label_x > XMAX_4D - 0.028:
            label_x = external_value - 0.012
            label_ha = 'right'
        else:
            label_ha = 'left'
        ax.text(label_x, y + 0.16, f'{external_value:.3f}', ha=label_ha, va='bottom', fontsize=11.0, color='#111111', fontweight='bold', zorder=7)
    ax.set_xlim(XMIN_4D, XMAX_4D)
    ax.set_ylim(-0.65, len(external_compare) - 0.35)
    ax.set_xlabel(spec['xlabel'])
    ax.set_title(spec['title'], color=endpoint_color, fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0)
axes[0].set_yticks(y_positions_4d)
axes[0].set_yticklabels(external_compare['row_label'])
for tick, short in zip(axes[0].get_yticklabels(), external_compare['short']):
    tick.set_color(MODALITY_COLORS[short])
    tick.set_fontsize(11.8)
for ax in axes[1:]:
    ax.tick_params(axis='y', labelleft=False)
legend_y = 0.945
fig.text(0.505, legend_y, '▬', color=ENDPOINT_COLORS['PFS'], fontsize=14.5, va='center')
fig.text(0.522, legend_y, 'External performance', fontsize=11.0, va='center')
fig.text(0.645, legend_y, '—■—', color='#333333', fontsize=11.6, va='center')
fig.text(0.684, legend_y, 'External 95% CI', fontsize=11.0, va='center')
fig.text(0.79, legend_y, '│', color=MODALITY_COLORS['M'], fontsize=14.5, va='center')
fig.text(0.801, legend_y, 'Internal CV mean', fontsize=11.0, va='center')
fig.subplots_adjust(left=0.24, right=0.988, top=0.84, bottom=0.13, wspace=0.26)
STEM_4D = OUT_DIR / get_value('science.paths.directory_112', 'directory_112')
fig.savefig(str(STEM_4D) + '.png', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_4D) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_4D) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_4D) + '.png')
print(str(STEM_4D) + '.pdf')
print(str(STEM_4D) + '.svg')


In [ ]:
source_4d_cols = ['model', 'short', 'display_name_4d', 'n_external', 'Response_AUC_mean', 'Response_AUC_sd', 'Response_AUROC', 'Response_AUROC_CI_lower', 'Response_AUROC_CI_upper', 'PFS_Cindex_mean', 'PFS_Cindex_sd', 'PFS_Cindex', 'PFS_Cindex_CI_lower', 'PFS_Cindex_CI_upper', 'OS_Cindex_mean', 'OS_Cindex_sd', 'OS_Cindex', 'OS_Cindex_CI_lower', 'OS_Cindex_CI_upper']
figure4d_source = external_compare[source_4d_cols].copy()
figure4d_source_path = OUT_DIR / get_value('science.tables.result_192', 'result_192.csv')
figure4d_source.to_csv(figure4d_source_path, index=False, encoding='utf-8-sig')





In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


In [ ]:
plt.rcParams.update({'font.family': 'Arial', 'font.size': 11.5, 'axes.titlesize': 13.0, 'axes.labelsize': 12.0, 'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5, 'legend.fontsize': 10.2, 'axes.linewidth': 0.95, 'xtick.major.width': 0.95, 'ytick.major.width': 0.95, 'xtick.major.size': 3.8, 'ytick.major.size': 3.8, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'axes.grid': False, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F'})
MODALITY_COLORS = {'M': '#177D72', 'R': '#C65F45', 'B': '#7252A3', 'U': '#4F7F3C'}
ENDPOINT_COLORS = {'Response': '#C94C4C', 'PFS': '#2F6FA3', 'OS': '#6A4C93'}
COLOR_TEXT = '#1F1F1F'
COLOR_MUTED = '#4A4A4A'
COLOR_GRID = '#E0E0E0'
COLOR_FULL = '#4A4A4A'
BG_MRU = '#EAF5F2'
BG_FULL = '#F2F2F2'
MRU_EDGE = '#0B5C53'
FIG_DPI = 600


In [ ]:
PROJECT_DIR = Path(str(project_path(get_value('science.paths.directory_008', 'directory_008'))))
STAGE2_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010') / get_value('science.paths.directory_011', 'directory_011')
if not STAGE2_DIR.exists():
    PROJECT_DIR = Path(str(project_path('')))
    STAGE2_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010') / get_value('science.paths.directory_011', 'directory_011')
SUMMARY_PATH = STAGE2_DIR / get_value('science.paths.directory_012', 'directory_012') / get_value('science.tables.metrics_009', 'metrics_009.csv')
SUPP_OUT_DIR = STAGE2_DIR / get_value('science.paths.directory_113', 'directory_113')
SUPP_OUT_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f'\nCannot find:\n{SUMMARY_PATH}\n\nPlease run the latest Stage2B combination analysis first.')
perf = pd.read_csv(SUMMARY_PATH, encoding='utf-8-sig')
required_cols = ['model', 'modalities', 'n_modalities', 'Response_AUPRC_mean', 'Response_AUPRC_sd']
missing = [c for c in required_cols if c not in perf.columns]
if missing:
    raise ValueError('Missing columns:\n' + '\n'.join(missing))
ABBR = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}

def short_modalities(x):
    return '+'.join((ABBR.get(m, m) for m in str(x).split('+')))
perf['short'] = perf['modalities'].apply(short_modalities)
perf['n_modalities'] = pd.to_numeric(perf['n_modalities']).astype(int)
MRU = 'M+R+U'
FULL = 'M+R+B+U'
if MRU not in set(perf['short']):
    raise ValueError('MRU was not found.')
if FULL not in set(perf['short']):
    raise ValueError('Full model was not found.')
plot_df = perf.sort_values(['Response_AUPRC_mean', 'n_modalities'], ascending=[False, True]).reset_index(drop=True)
display(plot_df[['short', 'n_modalities', 'Response_AUPRC_mean', 'Response_AUPRC_sd']])


In [ ]:
n_models = len(plot_df)
y = np.arange(n_models)[::-1]
fig, ax = plt.subplots(figsize=(8.8, 6.6), facecolor='white')
all_low = plot_df['Response_AUPRC_mean'] - plot_df['Response_AUPRC_sd']
all_high = plot_df['Response_AUPRC_mean'] + plot_df['Response_AUPRC_sd']
raw_min = float(all_low.min())
raw_max = float(all_high.max())
XMIN = max(0.4, np.floor((raw_min - 0.03) * 20) / 20)
XMAX = min(1.0, np.ceil((raw_max + 0.03) * 20) / 20)
ax.grid(False)
for pos, (_, row) in zip(y, plot_df.iterrows()):
    mean = float(row['Response_AUPRC_mean'])
    sd = float(row['Response_AUPRC_sd'])
    short = row['short']
    is_mru = short == MRU
    is_full = short == FULL
    if is_mru:
        ax.axhspan(pos - 0.42, pos + 0.42, color=BG_MRU, zorder=0.4)
    elif is_full:
        ax.axhspan(pos - 0.42, pos + 0.42, color=BG_FULL, zorder=0.4)
    bar_color = COLOR_FULL if is_full else ENDPOINT_COLORS['Response']
    bar_alpha = 0.88 if is_mru else 0.66
    if is_full:
        bar_alpha = 0.72
    ax.barh(pos, mean - XMIN, left=XMIN, height=0.34, color=bar_color, alpha=bar_alpha, edgecolor='none', zorder=2)
    lower = max(XMIN, mean - sd)
    upper = min(XMAX, mean + sd)
    ax.plot([lower, upper], [pos, pos], color=COLOR_TEXT, linewidth=0.95, alpha=0.82, zorder=4)
    ax.plot([lower, lower], [pos - 0.1, pos + 0.1], color=COLOR_TEXT, linewidth=0.8, alpha=0.82, zorder=4)
    ax.plot([upper, upper], [pos - 0.1, pos + 0.1], color=COLOR_TEXT, linewidth=0.8, alpha=0.82, zorder=4)
    if is_mru:
        marker = 'D'
        marker_size = 43
        edge = MRU_EDGE
        edge_width = 1.0
    elif is_full:
        marker = 's'
        marker_size = 39
        edge = '#333333'
        edge_width = 0.8
    else:
        marker = 'o'
        marker_size = 29
        edge = 'white'
        edge_width = 0.45
    ax.scatter(mean, pos, s=marker_size, marker=marker, facecolor=bar_color, edgecolor=edge, linewidth=edge_width, zorder=5)
    if mean + 0.04 < XMAX:
        ax.text(mean + 0.01, pos, f'{mean:.3f}', ha='left', va='center', fontsize=9.8, color=MRU_EDGE if is_mru else COLOR_TEXT, fontweight='bold' if is_mru else 'normal', zorder=6)
    else:
        ax.text(mean - 0.01, pos, f'{mean:.3f}', ha='right', va='center', fontsize=9.8, color=MRU_EDGE if is_mru else COLOR_TEXT, fontweight='bold' if is_mru else 'normal', zorder=6)
labels = []
for _, row in plot_df.iterrows():
    short = row['short']
    if short == MRU:
        labels.append(f'{short}   MRU')
    elif short == FULL:
        labels.append(f'{short}   Full')
    else:
        labels.append(short)
ax.set_yticks(y)
ax.set_yticklabels(labels)
for tick, (_, row) in zip(ax.get_yticklabels(), plot_df.iterrows()):
    short = row['short']
    if short == MRU:
        tick.set_color(MRU_EDGE)
        tick.set_fontweight('bold')
    elif short == FULL:
        tick.set_color(COLOR_FULL)
        tick.set_fontweight('bold')
    else:
        tick.set_color(COLOR_TEXT)
strip_x0 = XMIN - 0.041
strip_gap = 0.0105
strip_width = 0.008
for pos, (_, row) in zip(y, plot_df.iterrows()):
    present = row['short'].split('+')
    for j, m in enumerate(['M', 'R', 'B', 'U']):
        x0 = strip_x0 + j * strip_gap
        rect = Rectangle((x0, pos - 0.16), strip_width, 0.32, facecolor=MODALITY_COLORS[m] if m in present else '#ECECEC', edgecolor='none', alpha=1.0 if m in present else 0.75, clip_on=False, zorder=7)
        ax.add_patch(rect)
ax.set_xlim(XMIN, XMAX)
ax.set_ylim(-0.75, n_models - 0.25)
ax.set_xlabel('Response AUPRC')
ax.set_ylabel('')
ax.tick_params(axis='y', length=0, pad=37)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
fig.text(0.018, 0.985, 'S4A', ha='left', va='top', fontsize=13, fontweight='bold')
key_y = 0.947
key_x = 0.49
for i, m in enumerate(['M', 'R', 'B', 'U']):
    fig.text(key_x + i * 0.058, key_y, '■', color=MODALITY_COLORS[m], fontsize=10.2, va='center')
    fig.text(key_x + 0.012 + i * 0.058, key_y, m, fontsize=9.8, va='center', color=COLOR_TEXT)
fig.subplots_adjust(left=0.27, right=0.98, top=0.9, bottom=0.09)
STEM_S4A = SUPP_OUT_DIR / get_value('science.paths.directory_114', 'directory_114')
fig.savefig(str(STEM_S4A) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S4A) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S4A) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_S4A) + '.png')
print(str(STEM_S4A) + '.pdf')
print(str(STEM_S4A) + '.svg')


In [ ]:
source_s4a = plot_df[['model', 'modalities', 'short', 'n_modalities', 'Response_AUPRC_mean', 'Response_AUPRC_sd']].copy()
source_s4a['AUPRC_rank'] = np.arange(1, len(source_s4a) + 1)
source_s4a_path = SUPP_OUT_DIR / get_value('science.tables.result_193', 'result_193.csv')
source_s4a.to_csv(source_s4a_path, index=False, encoding='utf-8-sig')





In [ ]:
from sklearn.metrics import brier_score_loss

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None
STAGE2_PRED_PATH = first_existing([STAGE2_DIR / get_value('science.tables.predictions_004', 'predictions_004.csv')])
COMBO_PRED_PATH = first_existing([STAGE2_DIR / get_value('science.paths.directory_012', 'directory_012') / get_value('science.tables.predictions_005', 'predictions_005.csv'), STAGE2_DIR / get_value('science.paths.directory_044', 'directory_044') / get_value('science.tables.predictions_005', 'predictions_005.csv'), STAGE2_DIR / get_value('science.tables.predictions_005', 'predictions_005.csv')])
if STAGE2_PRED_PATH is None:
    raise FileNotFoundError(get_value('science.tables.predictions_027', 'predictions_027.csv'))
if COMBO_PRED_PATH is None:
    raise FileNotFoundError(get_value('science.tables.predictions_028', 'predictions_028.csv'))
prediction_required = ['sample', 'model', 'fold_id', 'Response_true', 'Response_pred']

def read_prediction_file(path):
    tmp = pd.read_csv(path, encoding='utf-8-sig')
    missing = [c for c in prediction_required if c not in tmp.columns]
    if missing:
        raise ValueError(f'{path}\nMissing columns: {missing}')
    return tmp
pred_stage2 = read_prediction_file(STAGE2_PRED_PATH)
pred_combo = read_prediction_file(COMBO_PRED_PATH)
pred_long = pd.concat([pred_stage2, pred_combo], ignore_index=True)
pred_long = pred_long.drop_duplicates(subset=['sample', 'model', 'fold_id'], keep='last').copy()
for col in ['Response_true', 'Response_pred']:
    pred_long[col] = pd.to_numeric(pred_long[col], errors='coerce')
pred_long = pred_long.dropna(subset=['sample', 'model', 'Response_true', 'Response_pred']).copy()


print('Combined rows      :', len(pred_long))
print('Models             :', pred_long['model'].nunique())


In [ ]:
best_single_row = perf[perf['n_modalities'] == 1].sort_values('Response_AUPRC_mean', ascending=False).iloc[0]
best_pair_row = perf[perf['n_modalities'] == 2].sort_values('Response_AUPRC_mean', ascending=False).iloc[0]
mru_row = perf[perf['short'] == MRU].iloc[0]
full_row = perf[perf['short'] == FULL].iloc[0]
selected_model_rows = pd.DataFrame([{'role': 'Best single', 'model': best_single_row['model'], 'short': best_single_row['short']}, {'role': 'Best pair', 'model': best_pair_row['model'], 'short': best_pair_row['short']}, {'role': 'MRU', 'model': mru_row['model'], 'short': mru_row['short']}, {'role': 'Full', 'model': full_row['model'], 'short': full_row['short']}])
selected_model_rows = selected_model_rows.drop_duplicates(subset=['model'], keep='first').reset_index(drop=True)
print('Selected models:')
display(selected_model_rows)
selected_model_names = set(selected_model_rows['model'])
pred_selected = pred_long[pred_long['model'].isin(selected_model_names)].copy()
label_check = pred_selected.groupby(['model', 'sample'])['Response_true'].nunique()
if (label_check > 1).any():
    bad = label_check[label_check > 1]
    raise ValueError('Inconsistent Response_true labels detected across repeated CV predictions:\n' + str(bad.head()))
patient_oof = pred_selected.groupby(['model', 'sample'], as_index=False).agg(Response_true=('Response_true', 'first'), Response_pred=('Response_pred', 'mean'), n_repeated_oof=('Response_pred', 'size'))
patient_oof = patient_oof.merge(selected_model_rows, on='model', how='left')
print('Patient-level averaged OOF predictions:')
display(patient_oof.groupby(['role', 'short'], as_index=False).agg(n_patients=('sample', 'nunique'), mean_repeated_oof=('n_repeated_oof', 'mean'), response_rate=('Response_true', 'mean')))


In [ ]:
N_CALIBRATION_BINS = 5

def wilson_interval(k, n, z=1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    p = k / n
    z2 = z ** 2
    center = (p + z2 / (2 * n)) / (1 + z2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z2 / (4 * n ** 2)) / (1 + z2 / n)
    return (max(0.0, center - half), min(1.0, center + half))

def make_calibration_bins(df_model, n_bins=5):
    d = df_model[['Response_true', 'Response_pred']].dropna().copy()
    try:
        d['cal_bin'] = pd.qcut(d['Response_pred'], q=n_bins, labels=False, duplicates='drop')
    except ValueError:
        d['cal_bin'] = pd.cut(d['Response_pred'], bins=n_bins, labels=False, include_lowest=True)
    records = []
    for bin_id, g in d.groupby('cal_bin', observed=True):
        n = len(g)
        k = int(g['Response_true'].sum())
        obs_rate = float(g['Response_true'].mean())
        mean_pred = float(g['Response_pred'].mean())
        lo, hi = wilson_interval(k, n)
        records.append({'cal_bin': int(bin_id), 'n': n, 'responders': k, 'mean_predicted_probability': mean_pred, 'observed_response_rate': obs_rate, 'observed_CI_lower': lo, 'observed_CI_upper': hi})
    return pd.DataFrame(records).sort_values('cal_bin').reset_index(drop=True)
calibration_parts = []
brier_records = []
for _, sel in selected_model_rows.iterrows():
    model_name = sel['model']
    d = patient_oof[patient_oof['model'] == model_name].copy()
    bins = make_calibration_bins(d, n_bins=N_CALIBRATION_BINS)
    bins['model'] = model_name
    bins['role'] = sel['role']
    bins['short'] = sel['short']
    calibration_parts.append(bins)
    brier = brier_score_loss(d['Response_true'], d['Response_pred'])
    brier_records.append({'model': model_name, 'role': sel['role'], 'short': sel['short'], 'n_patients': len(d), 'Response_Brier': float(brier)})
calibration_bins = pd.concat(calibration_parts, ignore_index=True)
brier_summary = pd.DataFrame(brier_records)
print('Brier score summary:')



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.2, 7.8), sharex=True, sharey=True, facecolor='white')
axes = axes.ravel()

def combination_accent(short):
    return ENDPOINT_COLORS['Response']
for ax, (_, sel) in zip(axes, selected_model_rows.iterrows()):
    model_name = sel['model']
    role = sel['role']
    short = sel['short']
    bins = calibration_bins[calibration_bins['model'] == model_name].copy()
    brier = float(brier_summary.loc[brier_summary['model'] == model_name, 'Response_Brier'].iloc[0])
    curve_color = combination_accent(short)
    ax.plot([0, 1], [0, 1], linestyle='--', color='#6B6B6B', linewidth=1.0, zorder=1)
    x = bins['mean_predicted_probability'].to_numpy(dtype=float)
    y_obs = bins['observed_response_rate'].to_numpy(dtype=float)
    y_lo = bins['observed_CI_lower'].to_numpy(dtype=float)
    y_hi = bins['observed_CI_upper'].to_numpy(dtype=float)
    yerr = np.vstack([y_obs - y_lo, y_hi - y_obs])
    ax.plot(x, y_obs, color=curve_color, linewidth=1.6, zorder=3)
    ax.errorbar(x, y_obs, yerr=yerr, fmt='o', markersize=5.4, markerfacecolor=curve_color, markeredgecolor='white', markeredgewidth=0.6, ecolor=curve_color, elinewidth=1.0, capsize=2.5, capthick=0.9, zorder=4)
    if role in ['MRU', 'Full']:
        title = f'{role}  |  {short}'
    else:
        title = f'{role}  |  {short}'
    title_color = MRU_EDGE if role == 'MRU' else COLOR_FULL if role == 'Full' else COLOR_TEXT
    ax.set_title(title, fontsize=11, fontweight='bold' if role in ['MRU', 'Full'] else 'normal', color=title_color, pad=8)
    ax.text(0.05, 0.92, f'Brier = {brier:.3f}', transform=ax.transAxes, ha='left', va='top', fontsize=10.2, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='#DDDDDD', linewidth=0.6, alpha=0.97))
    present = short.split('+')
    start_x = 0.73
    sig_y = 0.075
    dx = 0.055
    for j, m in enumerate(['M', 'R', 'B', 'U']):
        ax.scatter(start_x + j * dx, sig_y, transform=ax.transAxes, s=20, marker='s', facecolor=MODALITY_COLORS[m] if m in present else '#EAEAEA', edgecolor='none', clip_on=False, zorder=6)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks(np.arange(0, 1.01, 0.2))
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
for ax in axes[2:]:
    ax.set_xlabel('Mean predicted probability')
for ax in [axes[0], axes[2]]:
    ax.set_ylabel('Observed response rate')
fig.text(0.018, 0.985, 'S4B', ha='left', va='top', fontsize=13, fontweight='bold')
fig.subplots_adjust(left=0.11, right=0.98, top=0.91, bottom=0.09, hspace=0.3, wspace=0.25)
STEM_S4B = SUPP_OUT_DIR / get_value('science.paths.directory_115', 'directory_115')
fig.savefig(str(STEM_S4B) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S4B) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S4B) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_S4B) + '.png')
print(str(STEM_S4B) + '.pdf')
print(str(STEM_S4B) + '.svg')


In [ ]:
patient_oof_path = SUPP_OUT_DIR / get_value('science.tables.predictions_029', 'predictions_029.csv')
calibration_bins_path = SUPP_OUT_DIR / get_value('science.tables.result_194', 'result_194.csv')
brier_summary_path = SUPP_OUT_DIR / get_value('science.tables.metrics_069', 'metrics_069.csv')
patient_oof.to_csv(patient_oof_path, index=False, encoding='utf-8-sig')
calibration_bins.to_csv(calibration_bins_path, index=False, encoding='utf-8-sig')
brier_summary.to_csv(brier_summary_path, index=False, encoding='utf-8-sig')






In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, recall_score, precision_score, f1_score, balanced_accuracy_score, accuracy_score

def derive_threshold_from_patient_oof(df_model):
    y = pd.to_numeric(df_model['Response_true'], errors='coerce').to_numpy(dtype=float)
    p = pd.to_numeric(df_model['Response_pred'], errors='coerce').to_numpy(dtype=float)
    ok = np.isfinite(y) & np.isfinite(p)
    y = y[ok].astype(int)
    p = p[ok]
    if len(np.unique(y)) < 2:
        return np.nan
    fpr, tpr, thresholds = roc_curve(y, p)
    finite = np.isfinite(thresholds)
    if finite.sum() == 0:
        return np.nan
    youden = tpr[finite] - fpr[finite]
    return float(thresholds[finite][np.argmax(youden)])
threshold_records = []
for _, sel in selected_model_rows.iterrows():
    model_name = sel['model']
    d = patient_oof[patient_oof['model'] == model_name].copy()
    threshold = derive_threshold_from_patient_oof(d)
    threshold_records.append({'model': model_name, 'role': sel['role'], 'short': sel['short'], 'Response_threshold': threshold, 'n_patients': len(d)})
threshold_summary = pd.DataFrame(threshold_records)
print('Internally derived frozen OOF thresholds:')
display(threshold_summary)


In [ ]:
def response_metrics_at_threshold(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if tn + fp > 0 else np.nan
    ppv = precision_score(y_true, y_pred, zero_division=0)
    npv = tn / (tn + fn) if tn + fn > 0 else np.nan
    return {'Sensitivity': float(sensitivity), 'Specificity': float(specificity), 'PPV': float(ppv), 'NPV': float(npv), 'Balanced accuracy': float(balanced_accuracy_score(y_true, y_pred)), 'Accuracy': float(accuracy_score(y_true, y_pred)), 'F1': float(f1_score(y_true, y_pred, zero_division=0)), 'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)}
metric_records = []
for _, row in threshold_summary.iterrows():
    model_name = row['model']
    threshold = float(row['Response_threshold'])
    d = patient_oof[patient_oof['model'] == model_name].copy()
    result = response_metrics_at_threshold(d['Response_true'], d['Response_pred'], threshold)
    metric_records.append({'model': model_name, 'role': row['role'], 'short': row['short'], 'n_patients': len(d), 'Response_threshold': threshold, **result})
threshold_metrics = pd.DataFrame(metric_records)



In [ ]:
from matplotlib.colors import LinearSegmentedColormap
metric_order_s5 = ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'Balanced accuracy', 'F1']
plot_s5 = selected_model_rows.merge(threshold_metrics, on=['model', 'role', 'short'], how='left').copy()
heat_s5 = plot_s5[metric_order_s5].astype(float).to_numpy()
S5_CMAP = LinearSegmentedColormap.from_list('response_operating_metrics', ['#FAF5F4', '#F2D6D2', '#E8A6A0', ENDPOINT_COLORS['Response']], N=256)
fig, ax = plt.subplots(figsize=(10.5, 4.7), facecolor='white')
im = ax.imshow(heat_s5, cmap=S5_CMAP, vmin=0.45, vmax=1.0, aspect='auto', interpolation='nearest')
for i in range(heat_s5.shape[0]):
    for j in range(heat_s5.shape[1]):
        value = heat_s5[i, j]
        if not np.isfinite(value):
            txt = 'NA'
            text_color = COLOR_MUTED
        else:
            txt = f'{value:.2f}'
            text_color = 'white' if value >= 0.78 else COLOR_TEXT
        ax.text(j, i, txt, ha='center', va='center', fontsize=10.6, fontweight='bold' if plot_s5.iloc[i]['role'] in ['MRU', 'Full'] else 'normal', color=text_color, zorder=4)
ax.set_xticks(np.arange(len(metric_order_s5)))
ax.set_xticklabels(['Sensitivity', 'Specificity', 'PPV', 'NPV', 'Balanced\naccuracy', 'F1'])
ax.xaxis.tick_top()
ax.tick_params(axis='x', top=True, bottom=False, labeltop=True, labelbottom=False, length=0, pad=8)
row_labels_s5 = []
for _, row in plot_s5.iterrows():
    role = row['role']
    short = row['short']
    threshold = float(row['Response_threshold'])
    row_labels_s5.append(f'{role}  |  {short}\nthreshold={threshold:.3f}')
ax.set_yticks(np.arange(len(plot_s5)))
ax.set_yticklabels(row_labels_s5)
ax.tick_params(axis='y', length=0, pad=10)
for tick, (_, row) in zip(ax.get_yticklabels(), plot_s5.iterrows()):
    role = row['role']
    if role == 'MRU':
        tick.set_color(MRU_EDGE)
        tick.set_fontweight('bold')
    elif role == 'Full':
        tick.set_color(COLOR_FULL)
        tick.set_fontweight('bold')
    else:
        tick.set_color(COLOR_TEXT)
ax.set_xticks(np.arange(-0.5, len(metric_order_s5), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(plot_s5), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=2.2)
ax.tick_params(which='minor', bottom=False, left=False)
for i, (_, row) in enumerate(plot_s5.iterrows()):
    if row['role'] == 'MRU':
        rect = Rectangle((-0.5, i - 0.5), width=len(metric_order_s5), height=1.0, fill=False, edgecolor=MRU_EDGE, linewidth=1.8, zorder=5)
        ax.add_patch(rect)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.06, pad=0.18, aspect=40)
cbar.set_label('Operating-characteristic value', fontsize=10.6)
cbar.ax.tick_params(labelsize=7.3, length=2.5)
cbar.outline.set_visible(False)
fig.text(0.018, 0.985, 'S5', ha='left', va='top', fontsize=13, fontweight='bold')
fig.subplots_adjust(left=0.235, right=0.985, top=0.9, bottom=0.16)
STEM_S5 = SUPP_OUT_DIR / get_value('science.paths.directory_116', 'directory_116')
fig.savefig(str(STEM_S5) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S5) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S5) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_S5) + '.png')
print(str(STEM_S5) + '.pdf')
print(str(STEM_S5) + '.svg')


In [ ]:
s5_source_path = SUPP_OUT_DIR / get_value('science.tables.metrics_070', 'metrics_070.csv')
threshold_metrics.to_csv(s5_source_path, index=False, encoding='utf-8-sig')





In [ ]:
survival_required = ['PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']
missing_surv = [c for c in survival_required if c not in pred_long.columns]
if missing_surv:
    raise ValueError('Stage2 prediction files are missing survival columns:\n' + '\n'.join(missing_surv))
pred_surv_selected = pred_long[pred_long['model'].isin(set(selected_model_rows['model']))].copy()
for col in survival_required:
    pred_surv_selected[col] = pd.to_numeric(pred_surv_selected[col], errors='coerce')
for endpoint in ['PFS', 'OS']:
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    time_check = pred_surv_selected.groupby(['model', 'sample'])[time_col].nunique(dropna=True)
    status_check = pred_surv_selected.groupby(['model', 'sample'])[status_col].nunique(dropna=True)
    if (time_check > 1).any():
        raise ValueError(f'Inconsistent {endpoint}_time across repeated OOF predictions.')
    if (status_check > 1).any():
        raise ValueError(f'Inconsistent {endpoint}_status across repeated OOF predictions.')
patient_surv_oof = pred_surv_selected.groupby(['model', 'sample'], as_index=False).agg(PFS_time=('PFS_time', 'first'), PFS_status=('PFS_status', 'first'), PFS_risk=('PFS_risk', 'mean'), OS_time=('OS_time', 'first'), OS_status=('OS_status', 'first'), OS_risk=('OS_risk', 'mean'), n_repeated_oof=('fold_id', 'size'))
patient_surv_oof = patient_surv_oof.merge(selected_model_rows, on='model', how='left')
print('Patient-level aggregated OOF survival predictions:')
display(patient_surv_oof.groupby(['role', 'short'], as_index=False).agg(n_patients=('sample', 'nunique'), PFS_events=('PFS_status', 'sum'), OS_events=('OS_status', 'sum'), PFS_max=('PFS_time', 'max'), OS_max=('OS_time', 'max'), mean_repeated_oof=('n_repeated_oof', 'mean')))


In [ ]:
TIME_UNIT_LABEL = 'months'
PFS_AUC_TIMES = [6, 12, 18, 24]
OS_AUC_TIMES = [12, 18, 24, 36]
RUN_TD_AUC_BOOTSTRAP = True
N_TD_AUC_BOOTSTRAP = 500
TD_AUC_RANDOM_STATE = 20260809
try:
    from sksurv.util import Surv
    from sksurv.metrics import cumulative_dynamic_auc
except ImportError as e:
    raise ImportError('Figure S6 requires scikit-survival.\nRecommended installation in the multimodal conda environment:\nconda install -c conda-forge scikit-survival') from e

def clean_survival_model_data(df, endpoint):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    risk_col = f'{endpoint}_risk'
    d = df[['sample', time_col, status_col, risk_col]].dropna().copy()
    for col in [time_col, status_col, risk_col]:
        d[col] = pd.to_numeric(d[col], errors='coerce')
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    d = d[d[time_col] > 0].copy()
    d[status_col] = d[status_col].astype(int)
    return d

def estimable_times(d, endpoint, requested_times):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    event_times = d.loc[d[status_col] == 1, time_col]
    if len(event_times) == 0:
        return np.array([], dtype=float)
    min_event_time = float(event_times.min())
    max_followup = float(d[time_col].max())
    max_allowed = np.nextafter(max_followup, -np.inf)
    times = np.array([float(t) for t in requested_times if float(t) > min_event_time and float(t) < max_allowed], dtype=float)
    return times

def calculate_td_auc(d, endpoint, requested_times):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    risk_col = f'{endpoint}_risk'
    times = estimable_times(d, endpoint, requested_times)
    if len(times) == 0:
        raise ValueError(f'{endpoint}: none of the requested time points fall inside the estimable range.')
    y_surv = Surv.from_arrays(event=d[status_col].astype(bool).to_numpy(), time=d[time_col].astype(float).to_numpy())
    risk = d[risk_col].astype(float).to_numpy()
    auc_values, mean_auc = cumulative_dynamic_auc(y_surv, y_surv, risk, times)
    return (times, np.asarray(auc_values, dtype=float), float(mean_auc))


In [ ]:
td_auc_records = []
td_auc_mean_records = []
for endpoint, requested_times in [('PFS', PFS_AUC_TIMES), ('OS', OS_AUC_TIMES)]:
    for _, sel in selected_model_rows.iterrows():
        model_name = sel['model']
        role = sel['role']
        short = sel['short']
        d = clean_survival_model_data(patient_surv_oof[patient_surv_oof['model'] == model_name], endpoint)
        times, auc_values, mean_auc = calculate_td_auc(d, endpoint, requested_times)
        bootstrap_by_time = {float(t): [] for t in times}
        valid_boot = 0
        if RUN_TD_AUC_BOOTSTRAP:
            n = len(d)
            local_seed = TD_AUC_RANDOM_STATE + sum((ord(ch) for ch in str(model_name))) + (1000 if endpoint == 'OS' else 0)
            rng = np.random.default_rng(local_seed)
            for _ in range(N_TD_AUC_BOOTSTRAP):
                idx = rng.integers(0, n, size=n)
                db = d.iloc[idx].reset_index(drop=True)
                status_col = f'{endpoint}_status'
                time_col = f'{endpoint}_time'
                risk_col = f'{endpoint}_risk'
                if db[status_col].sum() == 0 or db[status_col].nunique() < 2:
                    continue
                try:
                    max_allowed_boot = np.nextafter(db[time_col].max(), -np.inf)
                    if np.any(times >= max_allowed_boot):
                        continue
                    yb = Surv.from_arrays(event=db[status_col].astype(bool).to_numpy(), time=db[time_col].astype(float).to_numpy())
                    rb = db[risk_col].astype(float).to_numpy()
                    auc_b, _ = cumulative_dynamic_auc(yb, yb, rb, times)
                    auc_b = np.asarray(auc_b, dtype=float)
                    if np.all(np.isfinite(auc_b)):
                        for t, val in zip(times, auc_b):
                            bootstrap_by_time[float(t)].append(float(val))
                        valid_boot += 1
                except Exception:
                    continue
        for t, auc_value in zip(times, auc_values):
            vals = np.asarray(bootstrap_by_time[float(t)], dtype=float)
            if len(vals) >= 30:
                ci_lo = float(np.quantile(vals, 0.025))
                ci_hi = float(np.quantile(vals, 0.975))
            else:
                ci_lo = np.nan
                ci_hi = np.nan
            td_auc_records.append({'endpoint': endpoint, 'role': role, 'model': model_name, 'short': short, 'time': float(t), 'time_unit': TIME_UNIT_LABEL, 'AUC': float(auc_value), 'AUC_CI_lower': ci_lo, 'AUC_CI_upper': ci_hi, 'n_patients': len(d), 'n_events': int(d[f'{endpoint}_status'].sum()), 'n_valid_bootstrap': int(len(vals))})
        td_auc_mean_records.append({'endpoint': endpoint, 'role': role, 'model': model_name, 'short': short, 'mean_AUC': mean_auc, 'n_patients': len(d), 'n_events': int(d[f'{endpoint}_status'].sum()), 'valid_bootstrap_complete': valid_boot})
td_auc_results = pd.DataFrame(td_auc_records)
td_auc_mean_summary = pd.DataFrame(td_auc_mean_records)
print('Time-dependent AUC point estimates:')

print('\nIntegrated/mean AUC summary returned by cumulative_dynamic_auc:')



In [ ]:
MODEL_VISUALS = {'Best single': {'linestyle': ':', 'marker': 'o', 'linewidth': 1.15, 'alpha': 0.72}, 'Best pair': {'linestyle': '--', 'marker': '^', 'linewidth': 1.25, 'alpha': 0.78}, 'MRU': {'linestyle': '-', 'marker': 'D', 'linewidth': 2.05, 'alpha': 1.0}, 'Full': {'linestyle': '-.', 'marker': 's', 'linewidth': 1.55, 'alpha': 0.92}}
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.7), sharey=True, facecolor='white')
for ax, endpoint in zip(axes, ['PFS', 'OS']):
    endpoint_color = ENDPOINT_COLORS[endpoint]
    d_endpoint = td_auc_results[td_auc_results['endpoint'] == endpoint].copy()
    ax.axhline(0.5, linestyle='--', color='#6B6B6B', linewidth=0.9, zorder=0)
    ax.grid(False)
    for _, sel in selected_model_rows.iterrows():
        role = sel['role']
        model_name = sel['model']
        short = sel['short']
        dm = d_endpoint[d_endpoint['model'] == model_name].sort_values('time').copy()
        if len(dm) == 0:
            continue
        vis = MODEL_VISUALS[role]
        curve_color = endpoint_color
        if role == 'Full':
            curve_color = COLOR_FULL
        if role == 'Best single':
            single_short = short.split('+')[0]
            curve_color = MODALITY_COLORS.get(single_short, endpoint_color)
        x = dm['time'].to_numpy(dtype=float)
        y_auc = dm['AUC'].to_numpy(dtype=float)
        lo = dm['AUC_CI_lower'].to_numpy(dtype=float)
        hi = dm['AUC_CI_upper'].to_numpy(dtype=float)
        valid_ci = np.isfinite(lo) & np.isfinite(hi)
        if valid_ci.any():
            ax.fill_between(x[valid_ci], lo[valid_ci], hi[valid_ci], color=curve_color, alpha=0.18 if role == 'MRU' else 0.045, linewidth=0, zorder=1)
        ax.plot(x, y_auc, linestyle=vis['linestyle'], marker=vis['marker'], markersize=5.3 if role == 'MRU' else 4.5, linewidth=vis['linewidth'], alpha=vis['alpha'], color=curve_color, markerfacecolor='white' if role in ['Best single', 'Best pair'] else curve_color, markeredgecolor=curve_color, markeredgewidth=0.9, label=f'{role} | {short}', zorder=5 if role == 'MRU' else 3)
    present_times = sorted(d_endpoint['time'].dropna().unique())
    ax.set_xticks(present_times)
    ax.set_xlabel(f'Time ({TIME_UNIT_LABEL})')
    ax.set_title(endpoint, color=endpoint_color, fontweight='bold', pad=9)
    ax.set_ylim(0.45, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[0].set_ylabel('Time-dependent AUC')
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.64, 0.935), ncol=2, frameon=False, fontsize=9.8, handlelength=2.4, columnspacing=1.3)
fig.text(0.018, 0.985, 'S6', ha='left', va='top', fontsize=13, fontweight='bold')
fig.subplots_adjust(left=0.09, right=0.985, top=0.82, bottom=0.13, wspace=0.22)
STEM_S6 = SUPP_OUT_DIR / get_value('science.paths.directory_117', 'directory_117')
fig.savefig(str(STEM_S6) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S6) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S6) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_S6) + '.png')
print(str(STEM_S6) + '.pdf')
print(str(STEM_S6) + '.svg')


In [ ]:
s6_point_path = SUPP_OUT_DIR / get_value('science.tables.result_195', 'result_195.csv')
s6_mean_path = SUPP_OUT_DIR / get_value('science.tables.metrics_071', 'metrics_071.csv')
s6_patient_path = SUPP_OUT_DIR / get_value('science.tables.result_196', 'result_196.csv')
td_auc_results.to_csv(s6_point_path, index=False, encoding='utf-8-sig')
td_auc_mean_summary.to_csv(s6_mean_path, index=False, encoding='utf-8-sig')
patient_surv_oof.to_csv(s6_patient_path, index=False, encoding='utf-8-sig')






In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
MODEL_RESULTS_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010')
EXTERNAL_BASE_DIR = MODEL_RESULTS_DIR / get_value('science.paths.directory_050', 'directory_050')
EXTERNAL_PRED_ALL_PATH = EXTERNAL_BASE_DIR / get_value('science.tables.predictions_019', 'predictions_019.csv.gz')
EXTERNAL_MODEL_ORDER = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
EXTERNAL_SHORT = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}
EXTERNAL_DISPLAY = {'Metagenomics': 'Metagenomics (KO)', 'Radiomics': 'CT radiomics', 'Blood': 'Blood metabolomics', 'Urine': 'Urine metabolomics'}

def read_external_predictions(path):
    d = pd.read_csv(path, encoding='utf-8-sig', compression='infer')
    required = ['model', 'Response_true', 'Response_pred']
    missing = [c for c in required if c not in d.columns]
    if missing:
        raise ValueError(f'{path}\nMissing required columns: {missing}')
    return d
if EXTERNAL_PRED_ALL_PATH.exists():
    external_pred_s7 = read_external_predictions(EXTERNAL_PRED_ALL_PATH)
    print('Loaded combined external prediction file:')
    
else:
    external_parts = []
    for model_name in EXTERNAL_MODEL_ORDER:
        pred_path = EXTERNAL_BASE_DIR / f'{model_name}_only' / get_value('science.tables.predictions_019', 'predictions_019.csv.gz')
        if pred_path.exists():
            tmp = read_external_predictions(pred_path)
            external_parts.append(tmp)
            print('Loaded:', pred_path)
    if len(external_parts) == 0:
        raise FileNotFoundError('No external patient-level prediction file was found. Please run the latest fixed external-validation notebook first.')
    external_pred_s7 = pd.concat(external_parts, ignore_index=True)
for col in ['Response_true', 'Response_pred']:
    external_pred_s7[col] = pd.to_numeric(external_pred_s7[col], errors='coerce')
external_pred_s7 = external_pred_s7.dropna(subset=['model', 'Response_true', 'Response_pred']).copy()
external_pred_s7 = external_pred_s7[external_pred_s7['model'].isin(EXTERNAL_MODEL_ORDER)].copy()
print('\nExternal models available:')
display(external_pred_s7.groupby('model', as_index=False).agg(n_external=('Response_true', 'size'), n_responders=('Response_true', 'sum'), response_rate=('Response_true', 'mean')))


In [ ]:
roc_parts_s7 = []
pr_parts_s7 = []
external_metric_records_s7 = []
for model_name in EXTERNAL_MODEL_ORDER:
    d = external_pred_s7[external_pred_s7['model'] == model_name].copy()
    if len(d) == 0:
        continue
    y = d['Response_true'].astype(int).to_numpy()
    p = d['Response_pred'].astype(float).to_numpy()
    if len(np.unique(y)) < 2:
        print(f'[Skipped] {model_name}: only one Response class.')
        continue
    fpr, tpr, _ = roc_curve(y, p)
    auroc = float(roc_auc_score(y, p))
    roc_parts_s7.append(pd.DataFrame({'model': model_name, 'short': EXTERNAL_SHORT[model_name], 'FPR': fpr, 'TPR': tpr}))
    precision, recall, _ = precision_recall_curve(y, p)
    auprc = float(average_precision_score(y, p))
    pr_parts_s7.append(pd.DataFrame({'model': model_name, 'short': EXTERNAL_SHORT[model_name], 'Recall': recall, 'Precision': precision}))
    external_metric_records_s7.append({'model': model_name, 'short': EXTERNAL_SHORT[model_name], 'display_name': EXTERNAL_DISPLAY[model_name], 'n_external': int(len(d)), 'n_responders': int(y.sum()), 'response_prevalence': float(y.mean()), 'Response_AUROC': auroc, 'Response_AUPRC': auprc})
roc_curve_s7 = pd.concat(roc_parts_s7, ignore_index=True)
pr_curve_s7 = pd.concat(pr_parts_s7, ignore_index=True)
external_metrics_s7 = pd.DataFrame(external_metric_records_s7)
external_metrics_s7['model_order'] = external_metrics_s7['model'].map({m: i for i, m in enumerate(EXTERNAL_MODEL_ORDER)})
external_metrics_s7 = external_metrics_s7.sort_values('model_order').reset_index(drop=True)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.7), facecolor='white')
ax_roc, ax_pr = axes
ax_roc.plot([0, 1], [0, 1], linestyle='--', color='#6B6B6B', linewidth=0.9, zorder=0)
for model_name in EXTERNAL_MODEL_ORDER:
    dcurve = roc_curve_s7[roc_curve_s7['model'] == model_name].copy()
    dmetric = external_metrics_s7[external_metrics_s7['model'] == model_name]
    if len(dcurve) == 0 or len(dmetric) == 0:
        continue
    m = dmetric.iloc[0]
    short = m['short']
    color = MODALITY_COLORS[short]
    label = f"{short}  n={int(m['n_external'])}  AUROC={m['Response_AUROC']:.3f}"
    ax_roc.plot(dcurve['FPR'], dcurve['TPR'], color=color, linewidth=1.8, alpha=0.98, label=label, zorder=3)
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1)
ax_roc.set_xlabel('False positive rate')
ax_roc.set_ylabel('True positive rate')
ax_roc.set_title('External ROC', color=ENDPOINT_COLORS['Response'], fontweight='bold', pad=9)
ax_roc.grid(False)
ax_roc.legend(loc='lower right', frameon=False, fontsize=9.8, handlelength=2.3)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)
for model_name in EXTERNAL_MODEL_ORDER:
    dcurve = pr_curve_s7[pr_curve_s7['model'] == model_name].copy()
    dmetric = external_metrics_s7[external_metrics_s7['model'] == model_name]
    if len(dcurve) == 0 or len(dmetric) == 0:
        continue
    m = dmetric.iloc[0]
    short = m['short']
    color = MODALITY_COLORS[short]
    label = f"{short}  n={int(m['n_external'])}  AUPRC={m['Response_AUPRC']:.3f}"
    ax_pr.plot(dcurve['Recall'], dcurve['Precision'], color=color, linewidth=1.8, alpha=0.98, label=label, zorder=3)
ax_pr.set_xlim(0, 1)
ax_pr.set_ylim(0, 1)
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title('External precision–recall', color=ENDPOINT_COLORS['Response'], fontweight='bold', pad=9)
ax_pr.grid(False)
ax_pr.legend(loc='lower left', frameon=False, fontsize=9.8, handlelength=2.3)
ax_pr.spines['top'].set_visible(False)
ax_pr.spines['right'].set_visible(False)
fig.text(0.018, 0.985, 'S7', ha='left', va='top', fontsize=13, fontweight='bold')
key_y = 0.936
key_x = 0.57
for i, m in enumerate(['M', 'R', 'B', 'U']):
    fig.text(key_x + i * 0.058, key_y, '■', color=MODALITY_COLORS[m], fontsize=10.2, va='center')
    fig.text(key_x + 0.012 + i * 0.058, key_y, m, fontsize=9.8, va='center', color=COLOR_TEXT)
fig.subplots_adjust(left=0.085, right=0.985, top=0.84, bottom=0.13, wspace=0.24)
STEM_S7 = SUPP_OUT_DIR / get_value('science.paths.directory_118', 'directory_118')
fig.savefig(str(STEM_S7) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S7) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S7) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()

print(str(STEM_S7) + '.png')
print(str(STEM_S7) + '.pdf')
print(str(STEM_S7) + '.svg')


In [ ]:
s7_roc_path = SUPP_OUT_DIR / get_value('science.tables.result_197', 'result_197.csv')
s7_pr_path = SUPP_OUT_DIR / get_value('science.tables.result_198', 'result_198.csv')
s7_metric_path = SUPP_OUT_DIR / get_value('science.tables.metrics_072', 'metrics_072.csv')
roc_curve_s7.to_csv(s7_roc_path, index=False, encoding='utf-8-sig')
pr_curve_s7.to_csv(s7_pr_path, index=False, encoding='utf-8-sig')
external_metrics_s7.to_csv(s7_metric_path, index=False, encoding='utf-8-sig')






In [ ]:
try:
    from lifelines import KaplanMeierFitter
except ImportError as e:
    raise ImportError('Figure S8 requires lifelines. Install with: pip install lifelines') from e
EXTERNAL_RISK_ALL_PATH = EXTERNAL_BASE_DIR / get_value('science.tables.result_048', 'result_048.csv')

def read_risk_summary(path):
    d = pd.read_csv(path, encoding='utf-8-sig')
    required = ['model', 'endpoint', 'frozen_cutoff', 'n_lower_risk', 'n_higher_risk', 'events_lower_risk', 'events_higher_risk', 'HR_higher_vs_lower', 'HR_CI_lower', 'HR_CI_upper', 'Cox_P', 'logrank_P']
    missing = [c for c in required if c not in d.columns]
    if missing:
        raise ValueError(f'{path}\\nMissing required columns: {missing}')
    return d
if EXTERNAL_RISK_ALL_PATH.exists():
    external_risk_s8 = read_risk_summary(EXTERNAL_RISK_ALL_PATH)
    
else:
    risk_parts = []
    for model_name in EXTERNAL_MODEL_ORDER:
        p = EXTERNAL_BASE_DIR / f'{model_name}_only' / get_value('science.tables.result_048', 'result_048.csv')
        if p.exists():
            risk_parts.append(read_risk_summary(p))
            print('Loaded:', p)
    if not risk_parts:
        raise FileNotFoundError('No frozen external risk-stratification summary was found. Run the latest fixed external-validation notebook first.')
    external_risk_s8 = pd.concat(risk_parts, ignore_index=True)
required_surv_cols_s8 = ['model', 'PFS_time', 'PFS_status', 'PFS_risk', 'PFS_risk_group_frozen', 'OS_time', 'OS_status', 'OS_risk', 'OS_risk_group_frozen']
missing_s8 = [c for c in required_surv_cols_s8 if c not in external_pred_s7.columns]
if missing_s8:
    raise ValueError('External patient-level prediction file is missing frozen survival columns:\\n' + '\\n'.join(missing_s8))
external_surv_s8 = external_pred_s7[required_surv_cols_s8].copy()
for col in ['PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']:
    external_surv_s8[col] = pd.to_numeric(external_surv_s8[col], errors='coerce')
external_risk_s8 = external_risk_s8[external_risk_s8['model'].isin(EXTERNAL_MODEL_ORDER)].copy()
external_risk_s8['model_order'] = external_risk_s8['model'].map({m: i for i, m in enumerate(EXTERNAL_MODEL_ORDER)})
external_risk_s8['endpoint_order'] = external_risk_s8['endpoint'].map({'PFS': 0, 'OS': 1})
external_risk_s8 = external_risk_s8.sort_values(['endpoint_order', 'model_order']).reset_index(drop=True)



In [ ]:
audit_records_s8 = []
for endpoint in ['PFS', 'OS']:
    risk_col = f'{endpoint}_risk'
    group_col = f'{endpoint}_risk_group_frozen'
    for model_name in EXTERNAL_MODEL_ORDER:
        d = external_surv_s8[external_surv_s8['model'] == model_name].dropna(subset=[risk_col, group_col]).copy()
        row = external_risk_s8[(external_risk_s8['model'] == model_name) & (external_risk_s8['endpoint'] == endpoint)]
        if len(d) == 0 or len(row) != 1:
            continue
        cutoff = float(row.iloc[0]['frozen_cutoff'])
        expected = np.where(d[risk_col].astype(float) >= cutoff, 'Higher risk', 'Lower risk')
        observed = d[group_col].astype(str).to_numpy()
        audit_records_s8.append({'model': model_name, 'endpoint': endpoint, 'frozen_cutoff': cutoff, 'n_patients_checked': len(d), 'n_group_mismatch': int(np.sum(expected != observed))})
s8_group_audit = pd.DataFrame(audit_records_s8)

if len(s8_group_audit) and (s8_group_audit['n_group_mismatch'] > 0).any():
    raise ValueError('Frozen risk-group audit failed: patient-level labels do not match stored internal cutoffs.')


In [ ]:
SURVIVAL_TIME_LABEL = 'Time'

def format_p_value(p):
    if not np.isfinite(p):
        return 'P=NA'
    if p < 0.001:
        return 'P<0.001'
    return f'P={p:.3f}'

def format_hr_annotation(row):
    hr = float(row['HR_higher_vs_lower'])
    lo = float(row['HR_CI_lower'])
    hi = float(row['HR_CI_upper'])
    p = float(row['logrank_P'])
    if np.isfinite(hr) and np.isfinite(lo) and np.isfinite(hi):
        hr_text = f'HR {hr:.2f} (95% CI {lo:.2f}–{hi:.2f})'
    else:
        hr_text = 'HR NA'
    return hr_text + '\\nlog-rank ' + format_p_value(p)
fig, axes = plt.subplots(2, 4, figsize=(14.8, 7.8), sharey=True, facecolor='white')
for row_idx, endpoint in enumerate(['PFS', 'OS']):
    endpoint_color = ENDPOINT_COLORS[endpoint]
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    group_col = f'{endpoint}_risk_group_frozen'
    for col_idx, model_name in enumerate(EXTERNAL_MODEL_ORDER):
        ax = axes[row_idx, col_idx]
        short = EXTERNAL_SHORT[model_name]
        display_name = EXTERNAL_DISPLAY[model_name]
        d = external_surv_s8[external_surv_s8['model'] == model_name].dropna(subset=[time_col, status_col, group_col]).copy()
        d = d[d[time_col] > 0].copy()
        summary_row = external_risk_s8[(external_risk_s8['model'] == model_name) & (external_risk_s8['endpoint'] == endpoint)]
        if len(d) == 0:
            ax.text(0.5, 0.5, 'No evaluable data', transform=ax.transAxes, ha='center', va='center', color=COLOR_MUTED, fontsize=10.6)
            ax.set_axis_off()
            continue
        kmf = KaplanMeierFitter()
        for group in ['Lower risk', 'Higher risk']:
            sub = d[d[group_col] == group].copy()
            if len(sub) == 0:
                continue
            kmf.fit(sub[time_col], event_observed=sub[status_col], label=f'{group} (n={len(sub)})')
            if group == 'Lower risk':
                kmf.plot_survival_function(ax=ax, ci_show=True, show_censors=True, color=endpoint_color, linestyle='-', linewidth=1.85, alpha=0.98, censor_styles={'ms': 3.2, 'marker': '|', 'mew': 0.9})
            else:
                kmf.plot_survival_function(ax=ax, ci_show=True, show_censors=True, color='#333333', linestyle='--', linewidth=1.55, alpha=0.97, censor_styles={'ms': 3.2, 'marker': '|', 'mew': 0.9})
        if len(summary_row) == 1:
            ax.text(0.04, 0.06, format_hr_annotation(summary_row.iloc[0]), transform=ax.transAxes, ha='left', va='bottom', fontsize=9.8, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.24', facecolor='white', edgecolor='#DDDDDD', linewidth=0.55, alpha=0.96))
        if row_idx == 0:
            ax.set_title(f'{short}  {display_name}', color=MODALITY_COLORS[short], fontsize=11, fontweight='bold', pad=8)
        ax.set_ylabel(f'{endpoint} probability' if col_idx == 0 else '')
        ax.set_xlabel(SURVIVAL_TIME_LABEL if row_idx == 1 else '')
        ax.set_ylim(0, 1.02)
        ax.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(frameon=False, fontsize=9.4, loc='upper right', handlelength=2.0, labelspacing=0.35, borderaxespad=0.25)
fig.text(0.018, 0.992, 'S8', ha='left', va='top', fontsize=13, fontweight='bold')
fig.subplots_adjust(left=0.065, right=0.995, top=0.94, bottom=0.1, hspace=0.28, wspace=0.22)
STEM_S8 = SUPP_OUT_DIR / get_value('science.paths.directory_119', 'directory_119')
for ext in ['png', 'pdf', 'svg']:
    kwargs = dict(bbox_inches='tight', facecolor='white')
    if ext == 'png':
        kwargs['dpi'] = FIG_DPI
    fig.savefig(str(STEM_S8) + f'.{ext}', **kwargs)
plt.show()

print(str(STEM_S8) + '.png')
print(str(STEM_S8) + '.pdf')
print(str(STEM_S8) + '.svg')


In [ ]:
s8_patient_path = SUPP_OUT_DIR / get_value('science.tables.result_199', 'result_199.csv')
s8_summary_path = SUPP_OUT_DIR / get_value('science.tables.metrics_073', 'metrics_073.csv')
s8_audit_path = SUPP_OUT_DIR / get_value('science.tables.result_200', 'result_200.csv')
external_surv_s8.to_csv(s8_patient_path, index=False, encoding='utf-8-sig')
external_risk_s8.to_csv(s8_summary_path, index=False, encoding='utf-8-sig')
s8_group_audit.to_csv(s8_audit_path, index=False, encoding='utf-8-sig')






In [ ]:
from sklearn.metrics import brier_score_loss
N_EXTERNAL_CALIBRATION_BINS = 5
if 'wilson_interval' not in globals():

    def wilson_interval(k, n, z=1.959963984540054):
        if n <= 0:
            return (np.nan, np.nan)
        p = k / n
        z2 = z ** 2
        center = (p + z2 / (2 * n)) / (1 + z2 / n)
        half = z * np.sqrt(p * (1 - p) / n + z2 / (4 * n ** 2)) / (1 + z2 / n)
        return (max(0.0, center - half), min(1.0, center + half))

def external_calibration_bins(df_model, n_bins=5):
    d = df_model[['Response_true', 'Response_pred']].dropna().copy()
    if len(d) == 0:
        return pd.DataFrame()
    try:
        d['cal_bin'] = pd.qcut(d['Response_pred'], q=n_bins, labels=False, duplicates='drop')
    except ValueError:
        d['cal_bin'] = pd.cut(d['Response_pred'], bins=n_bins, labels=False, include_lowest=True)
    rows = []
    for bin_id, g in d.groupby('cal_bin', observed=True):
        n = len(g)
        k = int(g['Response_true'].sum())
        obs = float(g['Response_true'].mean())
        pred = float(g['Response_pred'].mean())
        lo, hi = wilson_interval(k, n)
        rows.append({'cal_bin': int(bin_id), 'n': int(n), 'responders': int(k), 'mean_predicted_probability': pred, 'observed_response_rate': obs, 'observed_CI_lower': lo, 'observed_CI_upper': hi, 'calibration_deviation': obs - pred, 'deviation_CI_lower': lo - pred, 'deviation_CI_upper': hi - pred})
    return pd.DataFrame(rows).sort_values('mean_predicted_probability').reset_index(drop=True)
cal_parts = []
brier_rows = []
for model_name in EXTERNAL_MODEL_ORDER:
    d = external_pred_s7[external_pred_s7['model'] == model_name].dropna(subset=['Response_true', 'Response_pred']).copy()
    if len(d) == 0:
        continue
    d['Response_true'] = pd.to_numeric(d['Response_true'], errors='coerce')
    d['Response_pred'] = pd.to_numeric(d['Response_pred'], errors='coerce')
    d = d.dropna(subset=['Response_true', 'Response_pred'])
    bins = external_calibration_bins(d, n_bins=N_EXTERNAL_CALIBRATION_BINS)
    bins['model'] = model_name
    bins['short'] = EXTERNAL_SHORT[model_name]
    bins['display_name'] = EXTERNAL_DISPLAY[model_name]
    cal_parts.append(bins)
    y = d['Response_true'].astype(int).to_numpy()
    p = d['Response_pred'].astype(float).to_numpy()
    brier_rows.append({'model': model_name, 'short': EXTERNAL_SHORT[model_name], 'display_name': EXTERNAL_DISPLAY[model_name], 'n_external': int(len(d)), 'response_prevalence': float(np.mean(y)), 'Response_Brier': float(brier_score_loss(y, p))})
external_calibration_s9 = pd.concat(cal_parts, ignore_index=True)
external_brier_s9 = pd.DataFrame(brier_rows)




In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13.8, 4.3), sharey=True, facecolor='white')
for ax, model_name in zip(axes, EXTERNAL_MODEL_ORDER):
    short = EXTERNAL_SHORT[model_name]
    display_name = EXTERNAL_DISPLAY[model_name]
    color = MODALITY_COLORS[short]
    d = external_calibration_s9[external_calibration_s9['model'] == model_name].sort_values('mean_predicted_probability').copy()
    b = external_brier_s9[external_brier_s9['model'] == model_name]
    ax.axhline(0, color='#7F7F7F', linestyle='--', linewidth=0.9, zorder=0)
    ax.grid(False)
    if len(d) > 0:
        x = d['mean_predicted_probability'].to_numpy(float)
        y = d['calibration_deviation'].to_numpy(float)
        lo = d['deviation_CI_lower'].to_numpy(float)
        hi = d['deviation_CI_upper'].to_numpy(float)
        for xi, yi in zip(x, y):
            ax.plot([xi, xi], [0, yi], color=color, linewidth=1.05, alpha=0.82, zorder=1)
        ax.errorbar(x, y, yerr=np.vstack([y - lo, hi - y]), fmt='o', markersize=5.2, markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.6, ecolor=color, elinewidth=1.0, capsize=2.5, capthick=0.9, zorder=4)
    if len(b) == 1:
        r = b.iloc[0]
        ax.text(0.05, 0.95, f"Brier = {r['Response_Brier']:.3f}\nn = {int(r['n_external'])}\nResponse = {r['response_prevalence']:.1%}", transform=ax.transAxes, ha='left', va='top', fontsize=9.8, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='#DDDDDD', linewidth=0.55, alpha=0.97))
    ax.set_xlim(0, 1)
    ax.set_xlabel('Mean predicted probability')
    ax.set_title(f'{short}  {display_name}', color=color, fontsize=10.6, fontweight='bold', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[0].set_ylabel('Observed − predicted response rate')
all_lo = external_calibration_s9['deviation_CI_lower'].astype(float).dropna()
all_hi = external_calibration_s9['deviation_CI_upper'].astype(float).dropna()
if len(all_lo) and len(all_hi):
    y_abs = max(abs(float(all_lo.min())), abs(float(all_hi.max())), 0.1)
    y_lim = min(0.65, np.ceil((y_abs + 0.04) * 20) / 20)
else:
    y_lim = 0.4
for ax in axes:
    ax.set_ylim(-y_lim, y_lim)
fig.text(0.018, 0.988, 'S9', ha='left', va='top', fontsize=13, fontweight='bold')
fig.subplots_adjust(left=0.07, right=0.995, top=0.9, bottom=0.14, wspace=0.18)
STEM_S9 = SUPP_OUT_DIR / get_value('science.paths.directory_120', 'directory_120')
for ext in ['.png', '.pdf', '.svg']:
    kwargs = dict(bbox_inches='tight', facecolor='white')
    if ext == '.png':
        kwargs['dpi'] = FIG_DPI
    fig.savefig(str(STEM_S9) + ext, **kwargs)
plt.show()
print(str(STEM_S9) + '.png')
print(str(STEM_S9) + '.pdf')
print(str(STEM_S9) + '.svg')


In [ ]:
s9_bins_path = SUPP_OUT_DIR / get_value('science.tables.result_201', 'result_201.csv')
s9_brier_path = SUPP_OUT_DIR / get_value('science.tables.metrics_074', 'metrics_074.csv')
external_calibration_s9.to_csv(s9_bins_path, index=False, encoding='utf-8-sig')
external_brier_s9.to_csv(s9_brier_path, index=False, encoding='utf-8-sig')


